In [ ]:
import sys
from pathlib import Path
from datetime import date

# Walk up to repo root (folder containing strategies/) so the notebook
# runs from any cwd. Pre-2026-05 this looked for jplus/; that package
# was renamed to strategies/support/jplus_inputs.py during the P-300
# restructure, and the offline simulator helper now lives at
# studies/jplus_analytic/.
_p = Path.cwd()
while _p != _p.parent and not (_p / 'strategies').is_dir():
    _p = _p.parent
REPO_ROOT = _p
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# P2.6 consolidated the trader/dashboard split into prod.db; the
# 2026-05-16 reorg tucked it under data/databases/.
PROD_DB        = REPO_ROOT / 'data' / 'databases' / 'prod.db'
DB_PATH        = PROD_DB  # was data/dashboard.db
TRADER_DB_PATH = PROD_DB  # was data/trader.db

CAPITAL = 10000.0
VARIANT = 'p300_aggressive_v2_v1_0__replay_full2y'
START_D = date(2024, 5, 9)
END_D   = date(2026, 5, 8)

def metrics_from_eq_curve(curve):
    if not curve or len(curve) < 2:
        return 0.0, 0.0
    total_ret = (curve[-1] / CAPITAL - 1) * 100
    peak = CAPITAL; mdd = 0.0
    for eq in curve:
        peak = max(peak, eq)
        dd = (eq / peak - 1) * 100 if peak > 0 else 0
        mdd = min(mdd, dd)
    return total_ret, mdd

print(f'Repo root:   {REPO_ROOT}')
print(f'Dashboard:   {DB_PATH}  (exists={DB_PATH.exists()})')
print(f'Trader DB:   {TRADER_DB_PATH}  (exists={TRADER_DB_PATH.exists()})')
print(f'Variant:     {VARIANT}')
print(f'Window:      {START_D} → {END_D}, base capital ${CAPITAL:,.0f}')


In [3]:
import sqlite3, statistics

con = sqlite3.connect(str(DB_PATH))
con.row_factory = sqlite3.Row
trades = con.execute('''
    SELECT strategy, asset, direction, allocation_pct, leverage,
           pnl_usdt, pnl_pct,
           CAST((julianday(actual_exit_time) - julianday(actual_entry_time)) AS REAL) AS hold_days
    FROM trades
    WHERE strategy_variant = ?
      AND status = 'closed' AND pnl_usdt IS NOT NULL
''', (VARIANT,)).fetchall()
con.close()

by = {}
for t in trades:
    by.setdefault(t['strategy'], []).append(t)

order = sorted(by.keys(), key=lambda s: -sum(float(t['pnl_usdt']) for t in by[s]))
print(f'{"sleeve":<13} {"N":>4} {"L/S":>6} {"win%":>6} {"PF":>6} {"total_$":>11} {"avg_w":>8} {"avg_l":>8} {"best":>8} {"worst":>8} {"avg_h":>7}')
print('-' * 99)
for s in order:
    ts = by[s]
    pnls = [float(t['pnl_usdt']) for t in ts]
    wins = [p for p in pnls if p > 0]
    losses = [p for p in pnls if p <= 0]
    longs = sum(1 for t in ts if t['direction'] == 'LONG')
    shorts = sum(1 for t in ts if t['direction'] == 'SHORT')
    other = len(ts) - longs - shorts
    ls_str = f'{longs}/{shorts}' if other == 0 else f'{longs}/{shorts}/{other}'
    holds = [float(t['hold_days'] or 0) for t in ts]
    gw = sum(wins); gl = abs(sum(losses))
    pf = gw / gl if gl > 0 else float('inf')
    pf_str = 'inf' if pf == float('inf') else f'{pf:.2f}'
    avg_w = statistics.mean(wins) if wins else 0
    avg_l = statistics.mean(losses) if losses else 0
    print(f'{s:<13} {len(ts):>4} {ls_str:>6} {len(wins)/len(ts)*100:>5.1f}% {pf_str:>6} {sum(pnls):>+11,.2f} {avg_w:>+8.2f} {avg_l:>+8.2f} {max(pnls):>+8.2f} {min(pnls):>+8.2f} {statistics.mean(holds):>6.1f}d')

print()
print('Asset/direction split per sleeve:')
for s in order:
    ts = by[s]
    splits = {}
    for t in ts:
        key = f'{t["asset"]} {t["direction"]}'
        splits.setdefault(key, []).append(float(t['pnl_usdt']))
    parts = []
    for k, ps in sorted(splits.items()):
        wins = sum(1 for p in ps if p > 0)
        parts.append(f'{k}: {len(ps)} (wr {wins/len(ps)*100:.0f}%, ${sum(ps):+,.0f})')
    print(f'  {s}: ' + '  |  '.join(parts))


sleeve           N    L/S   win%     PF     total_$    avg_w    avg_l     best    worst   avg_h
---------------------------------------------------------------------------------------------------
ADX              8    4/4  50.0%   2.13   +2,923.88 +1376.57  -645.60 +1935.92  -795.29   34.4d
THU_BEAR        58   0/58  74.1%   4.95   +1,578.72   +46.00   -26.61  +196.38   -57.28    1.0d
CARRY            8    8/0  50.0%  12.25     +372.10  +101.29    -8.27  +187.20    -9.06   78.8d
FOMC             5    5/0 100.0%    inf     +183.20   +36.64    +0.00   +96.38   +11.46    0.5d
PDO_RETOUCH     44   44/0  54.5%   1.38      +37.98    +5.73    -4.97   +21.29   -23.37    0.2d
CPR             21   21/0  76.2%   1.31      +18.39    +4.82   -11.75   +16.93   -15.58    4.2d

Asset/direction split per sleeve:
  ADX: BTC LONG: 4 (wr 50%, $+1,727)  |  BTC SHORT: 4 (wr 50%, $+1,197)
  THU_BEAR: BTC SHORT: 29 (wr 72%, $+594)  |  ETH SHORT: 29 (wr 76%, $+984)
  CARRY: BTC LONG: 8 (wr 50%, $+372)
  FOMC: 

In [ ]:
import statistics
# 2026-05-14 restructure: jplus.simulate moved to studies.jplus_analytic.simulate.
# Same shape (simulate(start_date, end_date) → {date: {return_pct, mode, lev,
# ema_p, contrib_1x_pct fields, *_fired flags}}); same apply_r4_fees helper.
from studies.jplus_analytic import simulate as core_sim

s = core_sim.simulate(start_date=START_D.isoformat(), end_date=END_D.isoformat())
core_sim.apply_r4_fees(s)
days = sorted(s.keys())

def trade_stats(trades, label):
    """trades: list of dicts {dir, pnl_usdt, hold_days}"""
    if not trades:
        return label, 0, '0/0', 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
    pnls = [t['pnl_usdt'] for t in trades]
    wins = [p for p in pnls if p > 0]
    losses = [p for p in pnls if p <= 0]
    longs = sum(1 for t in trades if t['dir'] == 'LONG')
    shorts = sum(1 for t in trades if t['dir'] == 'SHORT')
    other = len(trades) - longs - shorts
    ls = f'{longs}/{shorts}' if other == 0 else f'{longs}/{shorts}/{other}'
    holds = [t['hold_days'] for t in trades]
    gw = sum(wins); gl = abs(sum(losses))
    pf = gw/gl if gl > 0 else float('inf')
    return (label, len(trades), ls,
            (len(wins)/len(trades)*100),
            pf,
            sum(pnls),
            statistics.mean(wins) if wins else 0,
            statistics.mean(losses) if losses else 0,
            max(pnls), min(pnls),
            statistics.mean(holds))

# EMA(BTC) — direction-run trades from ema_p
ema_trades = []
run_dir = None; run_compound = 1.0; run_days = 0
for d in days:
    rec = s[d]
    p = rec.get('ema_p', 0)
    direction = 'LONG' if p > 0 else ('SHORT' if p < 0 else None)
    contrib = rec.get('ema_contrib_1x_pct', 0) / 100.0
    if direction != run_dir:
        if run_dir is not None and run_days > 0:
            ema_trades.append({'dir': run_dir,
                               'pnl_usdt': CAPITAL * (run_compound - 1),
                               'hold_days': float(run_days)})
        run_dir = direction; run_compound = 1.0; run_days = 0
    if direction is not None:
        run_compound *= (1 + contrib)
        run_days += 1
if run_dir is not None and run_days > 0:
    ema_trades.append({'dir': run_dir,
                       'pnl_usdt': CAPITAL * (run_compound - 1),
                       'hold_days': float(run_days)})

# ETH daily — runs of consecutive non-zero-weight days; LONG-only
eth_trades = []
in_run = False; run_compound = 1.0; run_days = 0
for d in days:
    rec = s[d]
    contrib = rec.get('eth_daily_contrib_1x_pct', 0) / 100.0
    active = (rec.get('mode') in ('strong_bull', 'mild_bull'))
    if active:
        if not in_run:
            in_run = True; run_compound = 1.0; run_days = 0
        run_compound *= (1 + contrib)
        run_days += 1
    else:
        if in_run:
            eth_trades.append({'dir': 'LONG',
                               'pnl_usdt': CAPITAL * (run_compound - 1),
                               'hold_days': float(run_days)})
            in_run = False
if in_run:
    eth_trades.append({'dir': 'LONG',
                       'pnl_usdt': CAPITAL * (run_compound - 1),
                       'hold_days': float(run_days)})

# R4 sleeves — each fired day = 1 LONG intraday trade
def r4_trades(field_fired, field_contrib):
    out = []
    for d in days:
        rec = s[d]
        if rec.get(field_fired):
            out.append({'dir': 'LONG',
                        'pnl_usdt': CAPITAL * (rec.get(field_contrib, 0) / 100.0),
                        'hold_days': 0.25})
    return out

r4_btc    = r4_trades('r4_btc_fired',    'r4_btc_contrib_1x_pct')
r4_eth    = r4_trades('r4_eth_fired',    'r4_eth_contrib_1x_pct')
r4_btc_v2 = r4_trades('r4_btc_v2_fired', 'r4_btc_v2_contrib_1x_pct')
r4_eth_v2 = r4_trades('r4_eth_v2_fired', 'r4_eth_v2_contrib_1x_pct')

results = [
    trade_stats(ema_trades, 'EMA(BTC)'),
    trade_stats(eth_trades, 'ETH daily'),
    trade_stats(r4_btc,     'R4 BTC'),
    trade_stats(r4_eth,     'R4 ETH'),
    trade_stats(r4_btc_v2,  'R4 BTC v2'),
    trade_stats(r4_eth_v2,  'R4 ETH v2'),
]
results.sort(key=lambda r: -r[5])

print(f'{"sub-sleeve":<12} {"N":>4} {"L/S":>7} {"win%":>6} {"PF":>6} {"total_$":>11} {"avg_w":>9} {"avg_l":>9} {"best":>9} {"worst":>9} {"avg_h":>8}')
print('-' * 105)
for r in results:
    label, n, ls, winp, pf, total, aw, al, best, worst, avgh = r
    pf_s = 'inf' if pf == float('inf') else f'{pf:.2f}'
    print(f'{label:<12} {n:>4} {ls:>7} {winp:>5.1f}% {pf_s:>6} {total:>+11,.2f} {aw:>+9,.2f} {al:>+9,.2f} {best:>+9,.2f} {worst:>+9,.2f} {avgh:>7.1f}d')


In [ ]:
import sqlite3
from datetime import datetime, timedelta, timezone
from studies.jplus_analytic import simulate as core_sim

# Tactical sleeves: cash-settle PnL on exit-time date
con = sqlite3.connect(str(DB_PATH))
con.row_factory = sqlite3.Row
trades = con.execute('''
    SELECT strategy, actual_exit_time, pnl_usdt
    FROM trades WHERE strategy_variant = ?
      AND status = 'closed' AND pnl_usdt IS NOT NULL
''', (VARIANT,)).fetchall()
con.close()

sleeves = sorted({t['strategy'] for t in trades})
tactical_results = []
for sl in sleeves:
    daily_pnl = {}
    for t in trades:
        if t['strategy'] != sl:
            continue
        d = t['actual_exit_time'][:10]
        daily_pnl[d] = daily_pnl.get(d, 0) + float(t['pnl_usdt'])
    eq = CAPITAL; curve = []
    d = START_D
    while d <= END_D:
        eq += daily_pnl.get(d.isoformat(), 0)
        curve.append(eq)
        d += timedelta(days=1)
    ret, mdd = metrics_from_eq_curve(curve)
    tactical_results.append((sl, ret, mdd, curve[-1] - CAPITAL))

# Core sub-sleeves: compound 1x daily contribs
s = core_sim.simulate(start_date=START_D.isoformat(), end_date=END_D.isoformat())
core_sim.apply_r4_fees(s)
days = sorted(s.keys())

def core_metrics(field):
    eq = CAPITAL; curve = []
    for d in days:
        c = s[d].get(field, 0) / 100.0
        eq *= (1 + c)
        curve.append(eq)
    return metrics_from_eq_curve(curve), curve[-1] - CAPITAL

core_subs = [
    ('EMA(BTC)',  'ema_contrib_1x_pct'),
    ('ETH daily', 'eth_daily_contrib_1x_pct'),
    ('R4 BTC',    'r4_btc_contrib_1x_pct'),
    ('R4 ETH',    'r4_eth_contrib_1x_pct'),
    ('R4 BTC v2', 'r4_btc_v2_contrib_1x_pct'),
    ('R4 ETH v2', 'r4_eth_v2_contrib_1x_pct'),
]
core_results = []
for label, fld in core_subs:
    (ret, mdd), pnl_d = core_metrics(fld)
    core_results.append((label, ret, mdd, pnl_d))

# Core J+ as a whole + Combined portfolio (already published)
def variant_curve(variant_id):
    con = sqlite3.connect(str(DB_PATH))
    rows = con.execute('''
        SELECT date, return_1x_pct FROM variant_daily_returns
        WHERE variant_id = ? AND source = 'replay'
        ORDER BY date
    ''', (variant_id,)).fetchall()
    con.close()
    eq = CAPITAL; curve = []
    for _, r in rows:
        eq *= (1 + (r or 0) / 100)
        curve.append(eq)
    return curve

core_curve = variant_curve('p300_aggressive_v2_v1_0__core_full2y')
core_total = metrics_from_eq_curve(core_curve)
core_pnl   = core_curve[-1] - CAPITAL if core_curve else 0.0

combined_curve = variant_curve('p300_aggressive_v2_v1_0__full2y')
combined_total = metrics_from_eq_curve(combined_curve)
combined_pnl   = combined_curve[-1] - CAPITAL if combined_curve else 0.0

# BTC buy & hold
con = sqlite3.connect(str(TRADER_DB_PATH))
btc_rows = con.execute('SELECT timestamp, close FROM cd_spot_binance ORDER BY timestamp').fetchall()
con.close()
start_ts = int(datetime(START_D.year, START_D.month, START_D.day, tzinfo=timezone.utc).timestamp())
end_ts   = int(datetime(END_D.year,   END_D.month,   END_D.day,   tzinfo=timezone.utc).timestamp()) + 86400
btc_daily = {}
for ts, px in btc_rows:
    if ts < start_ts or ts >= end_ts:
        continue
    d = datetime.fromtimestamp(ts, tz=timezone.utc).date().isoformat()
    btc_daily[d] = float(px)
bd = sorted(btc_daily.items())
start_px = bd[0][1]
qty = CAPITAL / start_px
bh_curve = [qty * px for _, px in bd]
bh_total = metrics_from_eq_curve(bh_curve)
bh_pnl   = bh_curve[-1] - CAPITAL

# ── Print combined table ──
print('=' * 78)
print(f'  Per-sleeve standalone return + max DD over {START_D} to {END_D}')
print(f'  Capital base: ${CAPITAL:,.0f} each (tactical at native sleeve weight; Core sub-sleeves at 1x pre-vol-target)')
print('=' * 78)
print(f'  {"sleeve":<22} {"total_return":>14} {"max_DD":>10} {"P&L_$":>14}')
print('  ' + '-' * 74)
print(f'  {"TACTICAL SLEEVES":<22}')
for sl, ret, mdd, pnl in sorted(tactical_results, key=lambda x: -x[1]):
    print(f'    {sl:<20} {ret:>+13.2f}% {mdd:>+9.2f}% {pnl:>+14,.2f}')
print(f'  {"CORE SUB-SLEEVES (1x pre-vol-target)":<22}')
for label, ret, mdd, pnl in sorted(core_results, key=lambda x: -x[1]):
    print(f'    {label:<20} {ret:>+13.2f}% {mdd:>+9.2f}% {pnl:>+14,.2f}')
print(f'  {"AGGREGATES":<22}')
ret, mdd = core_total
print(f'    {"Core J+ (vol-targ.)":<20} {ret:>+13.2f}% {mdd:>+9.2f}% {core_pnl:>+14,.2f}')
ret, mdd = combined_total
print(f'    {"Combined portfolio":<20} {ret:>+13.2f}% {mdd:>+9.2f}% {combined_pnl:>+14,.2f}')
ret, mdd = bh_total
print(f'    {"BTC buy & hold":<20} {ret:>+13.2f}% {mdd:>+9.2f}% {bh_pnl:>+14,.2f}')
print('=' * 78)


In [6]:
import sqlite3, json

con = sqlite3.connect(str(DB_PATH))
spec = json.loads(con.execute(
    "SELECT spec_json FROM variants WHERE id = 'p300_aggressive_v2_v1_0'"
).fetchone()[0])
con.close()

print('sleeve_leverages:', spec.get('sleeve_leverages'))
print()
for s in spec.get('composition', []):
    if s.get('strategy_id') == 'CARRY':
        print('CARRY composition entry:')
        print(json.dumps(s, indent=2))


sleeve_leverages: {'core': 2.5, 's003': 5.0, 's078': 5.0, 's096': 5.0, 'pdo': 1.0, 'cpr': 1.0, 'fomc': 10.0, 'r4_btc': 1.0, 'r4_eth': 1.0, 'ema_btc': 1.0, 'eth_daily': 1.0}

